# Feature Engineering

This notebook creates a model-ready dataset by combining sales history
with store information, calendar features, holidays, promotions, and oil prices.

In [35]:
from pathlib import Path

import numpy as np
import pandas as pd


# Detect the project root directory
project_root = Path.cwd()

if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"

# Stop execution if the data directory cannot be found
if not raw_data_dir.exists():
    raise FileNotFoundError(
        f"Raw data directory was not found: {raw_data_dir}"
    )

print(f"Project root: {project_root}")
print(f"Raw data directory: {raw_data_dir}")

Project root: c:\Users\sasha\Projects\supply-chain-demand-forecast
Raw data directory: c:\Users\sasha\Projects\supply-chain-demand-forecast\data\raw


In [36]:
# Load historical sales
sales = pd.read_csv(
    raw_data_dir / "train.csv",
    parse_dates=["date"],
    dtype={
        "id": "int32",
        "store_nbr": "int16",
        "family": "category",
        "sales": "float32",
        "onpromotion": "int16"
    }
)

# Load store attributes
stores = pd.read_csv(
    raw_data_dir / "stores.csv",
    dtype={
        "store_nbr": "int16",
        "city": "category",
        "state": "category",
        "type": "category",
        "cluster": "int16"
    }
)

# Load oil prices
oil = pd.read_csv(
    raw_data_dir / "oil.csv",
    parse_dates=["date"],
    dtype={
        "dcoilwtico": "float32"
    }
)

# Load holiday and event information
holidays = pd.read_csv(
    raw_data_dir / "holidays_events.csv",
    parse_dates=["date"]
)

print(f"Sales rows: {len(sales):,}")
print(f"Stores rows: {len(stores):,}")
print(f"Oil rows: {len(oil):,}")
print(f"Holiday rows: {len(holidays):,}")

Sales rows: 3,000,888
Stores rows: 54
Oil rows: 1,218
Holiday rows: 350


In [37]:
# Add store attributes to every sales row
model_data = sales.merge(
    stores,
    on="store_nbr",
    how="left",
    validate="many_to_one",
    sort=False
)

# Validate that the merge did not change the number of rows
assert len(model_data) == len(sales)

# Validate that all stores received their attributes
assert model_data[
    ["city", "state", "type", "cluster"]
].notna().all().all()

print(f"Model data rows: {len(model_data):,}")
print(f"Model data columns: {model_data.shape[1]}")

model_data.head()

Model data rows: 3,000,888
Model data columns: 10


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13


In [38]:
# Extract calendar features from the date
model_data["year"] = model_data["date"].dt.year.astype("int16")
model_data["month"] = model_data["date"].dt.month.astype("int8")
model_data["day_of_month"] = model_data["date"].dt.day.astype("int8")
model_data["day_of_week"] = model_data["date"].dt.dayofweek.astype("int8")
model_data["week_of_year"] = (
    model_data["date"]
    .dt.isocalendar()
    .week
    .astype("int16")
)

# Mark Saturdays and Sundays
model_data["is_weekend"] = (
    model_data["day_of_week"] >= 5
).astype("int8")

calendar_columns = [
    "date",
    "year",
    "month",
    "day_of_month",
    "day_of_week",
    "week_of_year",
    "is_weekend"
]

model_data[calendar_columns].head(10)

,date,year,month,day_of_month,day_of_week,week_of_year,is_weekend
0,2013-01-01,2013,1,1,1,1,0
1,2013-01-01,2013,1,1,1,1,0
2,2013-01-01,2013,1,1,1,1,0
3,2013-01-01,2013,1,1,1,1,0
4,2013-01-01,2013,1,1,1,1,0
5,2013-01-01,2013,1,1,1,1,0
6,2013-01-01,2013,1,1,1,1,0
7,2013-01-01,2013,1,1,1,1,0
8,2013-01-01,2013,1,1,1,1,0
9,2013-01-01,2013,1,1,1,1,0


In [39]:
# Sort oil prices by date
oil = (
    oil
    .sort_values("date")
    .reset_index(drop=True)
)

# Calculate the expected number of calendar days
oil_calendar_days = (
    oil["date"].max() - oil["date"].min()
).days + 1

print(f"Rows: {len(oil):,}")
print(
    f"Date range: {oil['date'].min().date()} — "
    f"{oil['date'].max().date()}"
)
print(f"Calendar days in range: {oil_calendar_days:,}")
print(f"Observed dates: {oil['date'].nunique():,}")
print(f"Missing prices: {oil['dcoilwtico'].isna().sum():,}")
print(f"Duplicate dates: {oil['date'].duplicated().sum():,}")

oil.head(10)

Rows: 1,218
Date range: 2013-01-01 — 2017-08-31
Calendar days in range: 1,704
Observed dates: 1,218
Missing prices: 43
Duplicate dates: 0


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.139999
2,2013-01-03,92.970001
3,2013-01-04,93.120003
4,2013-01-07,93.199997
5,2013-01-08,93.209999
6,2013-01-09,93.080002
7,2013-01-10,93.809998
8,2013-01-11,93.599998
9,2013-01-14,94.269997


In [40]:
# Create a complete daily calendar for the sales period
daily_dates = pd.DataFrame({
    "date": pd.date_range(
        start=model_data["date"].min(),
        end=model_data["date"].max(),
        freq="D"
    )
})

# Add the available oil prices
oil_daily = daily_dates.merge(
    oil[["date", "dcoilwtico"]],
    on="date",
    how="left",
    validate="one_to_one"
)

missing_before_fill = oil_daily["dcoilwtico"].isna().sum()

# Fill missing dates with the latest known price
oil_daily["oil_price"] = (
    oil_daily["dcoilwtico"]
    .ffill()
    .astype("float32")
)

missing_after_fill = oil_daily["oil_price"].isna().sum()

# Keep only the final columns
oil_daily = oil_daily[["date", "oil_price"]]

print(f"Daily oil rows: {len(oil_daily):,}")
print(f"Missing dates before filling: {missing_before_fill:,}")
print(f"Missing dates after filling: {missing_after_fill:,}")

oil_daily.head(10)

Daily oil rows: 1,688
Missing dates before filling: 525
Missing dates after filling: 1


,date,oil_price
0,2013-01-01,NaN
1,2013-01-02,93.139999
2,2013-01-03,92.970001
3,2013-01-04,93.120003
4,2013-01-05,93.120003
5,2013-01-06,93.120003
6,2013-01-07,93.199997
7,2013-01-08,93.209999
8,2013-01-09,93.080002
9,2013-01-10,93.809998


In [41]:
# Save the row count before merging
rows_before_oil_merge = len(model_data)

# Add daily oil prices to the model dataset
model_data = model_data.merge(
    oil_daily,
    on="date",
    how="left",
    validate="many_to_one",
    sort=False
)

# Validate the merge result
assert len(model_data) == rows_before_oil_merge

print(f"Rows after merge: {len(model_data):,}")
print(f"Columns after merge: {model_data.shape[1]}")
print(
    f"Rows without oil price: "
    f"{model_data['oil_price'].isna().sum():,}"
)

model_data[
    ["date", "store_nbr", "family", "sales", "oil_price"]
].head()

Rows after merge: 3,000,888
Columns after merge: 17
Rows without oil price: 1,782


,date,store_nbr,family,sales,oil_price
0,2013-01-01,1,AUTOMOTIVE,0.0,NaN
1,2013-01-01,1,BABY CARE,0.0,NaN
2,2013-01-01,1,BEAUTY,0.0,NaN
3,2013-01-01,1,BEVERAGES,0.0,NaN
4,2013-01-01,1,BOOKS,0.0,NaN


In [42]:
# Sort holiday records by date
holidays = (
    holidays
    .sort_values("date")
    .reset_index(drop=True)
)

# Keep actual holidays and exclude transferred original dates
effective_holidays = holidays.loc[
    holidays["type"].isin(
        ["Holiday", "Additional", "Transfer", "Bridge"]
    )
    & (holidays["transferred"] == False)
].copy()

# Keep public events separately
events = holidays.loc[
    holidays["type"] == "Event"
].copy()

# Keep special working days separately
special_work_days = holidays.loc[
    holidays["type"] == "Work Day"
].copy()

print(f"Original holiday records: {len(holidays):,}")
print(f"Effective holiday records: {len(effective_holidays):,}")
print(f"Event records: {len(events):,}")
print(f"Special work-day records: {len(special_work_days):,}")

print("\nEffective holidays by locale:")
print(effective_holidays["locale"].value_counts())

Original holiday records: 350
Effective holiday records: 277
Event records: 56
Special work-day records: 5

Effective holidays by locale:
locale
Local       148
National    105
Regional     24
Name: count, dtype: int64


In [43]:
# Create one row for each store and observed date
store_calendar = (
    model_data[
        ["date", "store_nbr", "city", "state"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Use the same text type for geographic matching
store_calendar["city"] = store_calendar["city"].astype("string")
store_calendar["state"] = store_calendar["state"].astype("string")

print(f"Store-calendar rows: {len(store_calendar):,}")
print(
    f"Duplicate store-date rows: "
    f"{store_calendar.duplicated(['date', 'store_nbr']).sum():,}"
)

store_calendar.head()

Store-calendar rows: 90,936
Duplicate store-date rows: 0


,date,store_nbr,city,state
0,2013-01-01,1,Quito,Pichincha
1,2013-01-01,10,Quito,Pichincha
2,2013-01-01,11,Cayambe,Pichincha
3,2013-01-01,12,Latacunga,Cotopaxi
4,2013-01-01,13,Latacunga,Cotopaxi


In [44]:
# Prepare national holiday dates
national_holidays = (
    effective_holidays.loc[
        effective_holidays["locale"] == "National",
        ["date"]
    ]
    .drop_duplicates()
    .assign(is_national_holiday=1)
)

# Prepare regional holiday dates and states
regional_holidays = (
    effective_holidays.loc[
        effective_holidays["locale"] == "Regional",
        ["date", "locale_name"]
    ]
    .rename(columns={"locale_name": "state"})
    .drop_duplicates()
    .assign(is_regional_holiday=1)
)

regional_holidays["state"] = (
    regional_holidays["state"].astype("string")
)

# Prepare local holiday dates and cities
local_holidays = (
    effective_holidays.loc[
        effective_holidays["locale"] == "Local",
        ["date", "locale_name"]
    ]
    .rename(columns={"locale_name": "city"})
    .drop_duplicates()
    .assign(is_local_holiday=1)
)

local_holidays["city"] = (
    local_holidays["city"].astype("string")
)

In [45]:
# Match national holidays by date
store_calendar = store_calendar.merge(
    national_holidays,
    on="date",
    how="left",
    validate="many_to_one"
)

# Match regional holidays by date and state
store_calendar = store_calendar.merge(
    regional_holidays,
    on=["date", "state"],
    how="left",
    validate="many_to_one"
)

# Match local holidays by date and city
store_calendar = store_calendar.merge(
    local_holidays,
    on=["date", "city"],
    how="left",
    validate="many_to_one"
)

holiday_columns = [
    "is_national_holiday",
    "is_regional_holiday",
    "is_local_holiday"
]

# Replace missing matches with zero
store_calendar[holiday_columns] = (
    store_calendar[holiday_columns]
    .fillna(0)
    .astype("int8")
)

# Mark a date as a holiday if any holiday level applies
store_calendar["is_holiday"] = (
    store_calendar[holiday_columns]
    .max(axis=1)
    .astype("int8")
)

print(f"Store-calendar rows: {len(store_calendar):,}")
print(
    f"Holiday store-days: "
    f"{store_calendar['is_holiday'].sum():,}"
)
print(
    f"Duplicate store-date rows: "
    f"{store_calendar.duplicated(['date', 'store_nbr']).sum():,}"
)

store_calendar.loc[
    store_calendar["is_holiday"] == 1
].head(10)

Store-calendar rows: 90,936
Holiday store-days: 4,598
Duplicate store-date rows: 0


,date,store_nbr,city,state,is_national_holiday,is_regional_holiday,is_local_holiday,is_holiday
0,2013-01-01,1,Quito,Pichincha,1,0,0,1
1,2013-01-01,10,Quito,Pichincha,1,0,0,1
2,2013-01-01,11,Cayambe,Pichincha,1,0,0,1
3,2013-01-01,12,Latacunga,Cotopaxi,1,0,0,1
4,2013-01-01,13,Latacunga,Cotopaxi,1,0,0,1
5,2013-01-01,14,Riobamba,Chimborazo,1,0,0,1
6,2013-01-01,15,Ibarra,Imbabura,1,0,0,1
7,2013-01-01,16,Santo Domingo,Santo Domingo de los Tsachilas,1,0,0,1
8,2013-01-01,17,Quito,Pichincha,1,0,0,1
9,2013-01-01,18,Quito,Pichincha,1,0,0,1


In [46]:
# Confirm that events and special work days are national
print("Event locales:")
print(events["locale"].value_counts())

print("\nSpecial work-day locales:")
print(special_work_days["locale"].value_counts())

Event locales:
locale
National    56
Name: count, dtype: int64

Special work-day locales:
locale
National    5
Name: count, dtype: int64


In [47]:
# Prepare unique event dates
event_dates = (
    events[["date"]]
    .drop_duplicates()
    .assign(is_event=1)
)

# Prepare unique special work-day dates
special_work_dates = (
    special_work_days[["date"]]
    .drop_duplicates()
    .assign(is_special_work_day=1)
)

# Match events by date
store_calendar = store_calendar.merge(
    event_dates,
    on="date",
    how="left",
    validate="many_to_one"
)

# Match special work days by date
store_calendar = store_calendar.merge(
    special_work_dates,
    on="date",
    how="left",
    validate="many_to_one"
)

# Replace missing matches with zero
store_calendar[
    ["is_event", "is_special_work_day"]
] = (
    store_calendar[
        ["is_event", "is_special_work_day"]
    ]
    .fillna(0)
    .astype("int8")
)

print(
    f"Event store-days: "
    f"{store_calendar['is_event'].sum():,}"
)
print(
    f"Special work store-days: "
    f"{store_calendar['is_special_work_day'].sum():,}"
)
print(
    f"Duplicate store-date rows: "
    f"{store_calendar.duplicated(['date', 'store_nbr']).sum():,}"
)

Event store-days: 2,970
Special work store-days: 270
Duplicate store-date rows: 0


In [48]:
# Select store-date features required for modeling
store_date_features = store_calendar[
    [
        "date",
        "store_nbr",
        "is_national_holiday",
        "is_regional_holiday",
        "is_local_holiday",
        "is_holiday",
        "is_event",
        "is_special_work_day"
    ]
].copy()

rows_before_calendar_merge = len(model_data)

# Add store-level calendar features
model_data = model_data.merge(
    store_date_features,
    on=["date", "store_nbr"],
    how="left",
    validate="many_to_one",
    sort=False
)

# Validate the merge
assert len(model_data) == rows_before_calendar_merge

calendar_flag_columns = [
    "is_national_holiday",
    "is_regional_holiday",
    "is_local_holiday",
    "is_holiday",
    "is_event",
    "is_special_work_day"
]

assert model_data[
    calendar_flag_columns
].notna().all().all()

print(f"Model rows: {len(model_data):,}")
print(f"Model columns: {model_data.shape[1]}")
print(
    f"Missing calendar flags: "
    f"{model_data[calendar_flag_columns].isna().sum().sum():,}"
)

model_data[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "is_holiday",
        "is_event",
        "is_special_work_day"
    ]
].head(10)

Model rows: 3,000,888
Model columns: 23
Missing calendar flags: 0


,date,store_nbr,family,sales,is_holiday,is_event,is_special_work_day
0,2013-01-01,1,AUTOMOTIVE,0.0,1,0,0
1,2013-01-01,1,BABY CARE,0.0,1,0,0
2,2013-01-01,1,BEAUTY,0.0,1,0,0
3,2013-01-01,1,BEVERAGES,0.0,1,0,0
4,2013-01-01,1,BOOKS,0.0,1,0,0
5,2013-01-01,1,BREAD/BAKERY,0.0,1,0,0
6,2013-01-01,1,CELEBRATION,0.0,1,0,0
7,2013-01-01,1,CLEANING,0.0,1,0,0
8,2013-01-01,1,DAIRY,0.0,1,0,0
9,2013-01-01,1,DELI,0.0,1,0,0


In [49]:
# Load only test dates to determine the forecast horizon
test_dates = (
    pd.read_csv(
        raw_data_dir / "test.csv",
        usecols=["date"],
        parse_dates=["date"]
    )["date"]
    .drop_duplicates()
    .sort_values()
)

forecast_horizon = test_dates.nunique()

print(f"Test start: {test_dates.min().date()}")
print(f"Test end: {test_dates.max().date()}")
print(f"Forecast horizon: {forecast_horizon} days")

Test start: 2017-08-16
Test end: 2017-08-31
Forecast horizon: 16 days


In [50]:
# Select the last observed dates for validation
validation_dates = (
    model_data["date"]
    .drop_duplicates()
    .sort_values()
    .tail(forecast_horizon)
)

validation_start = validation_dates.min()
validation_end = validation_dates.max()

# Create memory-efficient Boolean masks
validation_mask = model_data["date"].isin(validation_dates)
training_mask = ~validation_mask

# Confirm that training data ends before validation
assert (
    model_data.loc[training_mask, "date"].max()
    < validation_start
)

print(
    f"Training period: "
    f"{model_data.loc[training_mask, 'date'].min().date()} — "
    f"{model_data.loc[training_mask, 'date'].max().date()}"
)

print(
    f"Validation period: "
    f"{validation_start.date()} — "
    f"{validation_end.date()}"
)

print(
    f"Training rows: {training_mask.sum():,}"
)

print(
    f"Validation rows: {validation_mask.sum():,}"
)

Training period: 2013-01-01 — 2017-07-30
Validation period: 2017-07-31 — 2017-08-15
Training rows: 2,972,376
Validation rows: 28,512


In [51]:
# Calculate the oil median using training dates only
training_oil_median = (
    oil_daily.loc[
        oil_daily["date"] < validation_start,
        "oil_price"
    ]
    .median()
)

# Remember which rows originally had no oil price
model_data["oil_price_missing"] = (
    model_data["oil_price"]
    .isna()
    .astype("int8")
)

# Fill the remaining missing value
model_data["oil_price"] = (
    model_data["oil_price"]
    .fillna(training_oil_median)
    .astype("float32")
)

print(f"Training oil median: {training_oil_median:.2f}")
print(
    f"Missing oil prices after filling: "
    f"{model_data['oil_price'].isna().sum():,}"
)

Training oil median: 53.61
Missing oil prices after filling: 0


In [52]:
# Confirm that the rows are ordered chronologically
assert model_data["date"].is_monotonic_increasing

# Group sales into separate store-family time series
sales_by_store_family = model_data.groupby(
    ["store_nbr", "family"],
    observed=True,
    sort=False
)["sales"]

# Add historical sales features
model_data["sales_lag_16"] = (
    sales_by_store_family
    .shift(16)
    .astype("float32")
)

model_data["sales_lag_21"] = (
    sales_by_store_family
    .shift(21)
    .astype("float32")
)

model_data["sales_lag_28"] = (
    sales_by_store_family
    .shift(28)
    .astype("float32")
)

lag_columns = [
    "sales_lag_16",
    "sales_lag_21",
    "sales_lag_28"
]

print("Missing lag values in the complete dataset:")
print(model_data[lag_columns].isna().sum())

print("\nMissing lag values in validation:")
print(
    model_data.loc[
        validation_mask,
        lag_columns
    ].isna().sum()
)

Missing lag values in the complete dataset:
sales_lag_16    28512
sales_lag_21    37422
sales_lag_28    49896
dtype: int64

Missing lag values in validation:
sales_lag_16    0
sales_lag_21    0
sales_lag_28    0
dtype: int64


In [53]:
# Inspect lag features for one validation series
lag_example = model_data.loc[
    validation_mask
    & (model_data["store_nbr"] == 1)
    & (model_data["family"] == "BEVERAGES"),
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "sales_lag_16",
        "sales_lag_21",
        "sales_lag_28"
    ]
]

lag_example.head(10)

,date,store_nbr,family,sales,sales_lag_16,sales_lag_21,sales_lag_28
2972379,2017-07-31,1,BEVERAGES,2414.0,2183.0,2338.0,2526.0
2974161,2017-08-01,1,BEVERAGES,2627.0,1079.0,2372.0,2284.0
2975943,2017-08-02,1,BEVERAGES,2645.0,2381.0,2460.0,2691.0
2977725,2017-08-03,1,BEVERAGES,2037.0,2589.0,2115.0,2284.0
2979507,2017-08-04,1,BEVERAGES,2479.0,2369.0,2529.0,2574.0
2981289,2017-08-05,1,BEVERAGES,2093.0,2006.0,2183.0,2351.0
2983071,2017-08-06,1,BEVERAGES,968.0,2859.0,1079.0,1181.0
2984853,2017-08-07,1,BEVERAGES,2086.0,2258.0,2381.0,2338.0
2986635,2017-08-08,1,BEVERAGES,2418.0,1024.0,2589.0,2372.0
2988417,2017-08-09,1,BEVERAGES,2311.0,2158.0,2369.0,2460.0


In [54]:
# Use only history available before the 16-day forecast horizon
safe_sales_history = model_data["sales_lag_16"]

# Group the safe history by store and family
safe_history_by_series = safe_sales_history.groupby(
    [
        model_data["store_nbr"],
        model_data["family"]
    ],
    observed=True,
    sort=False
)

# Calculate the average of the latest 7 available historical values
model_data["sales_rolling_mean_7"] = (
    safe_history_by_series
    .transform(
        lambda values: values.rolling(
            window=7,
            min_periods=1
        ).mean()
    )
    .astype("float32")
)

# Calculate the average of the latest 28 available historical values
model_data["sales_rolling_mean_28"] = (
    safe_history_by_series
    .transform(
        lambda values: values.rolling(
            window=28,
            min_periods=1
        ).mean()
    )
    .astype("float32")
)

rolling_columns = [
    "sales_rolling_mean_7",
    "sales_rolling_mean_28"
]

print("Missing rolling values in validation:")
print(
    model_data.loc[
        validation_mask,
        rolling_columns
    ].isna().sum()
)

Missing rolling values in validation:
sales_rolling_mean_7     0
sales_rolling_mean_28    0
dtype: int64


In [55]:
# Inspect rolling features for one validation series
rolling_example = model_data.loc[
    validation_mask
    & (model_data["store_nbr"] == 1)
    & (model_data["family"] == "BEVERAGES"),
    [
        "date",
        "sales",
        "sales_lag_16",
        "sales_lag_21",
        "sales_lag_28",
        "sales_rolling_mean_7",
        "sales_rolling_mean_28"
    ]
]

rolling_example.head(10).round(1)

C:\Users\sasha\AppData\Local\Temp\ipykernel_12444\710594699.py:17: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  rolling_example.head(10).round(1)


,date,sales,sales_lag_16,sales_lag_21,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_28
2972379,2017-07-31,2414.0,2183.0,2338.0,2526.0,2168.300049,2207.500000
2974161,2017-08-01,2627.0,1079.0,2372.0,2284.0,2153.699951,2210.100098
2975943,2017-08-02,2645.0,2381.0,2460.0,2691.0,2159.899902,2206.100098
2977725,2017-08-03,2037.0,2589.0,2115.0,2284.0,2190.899902,2192.199951
2979507,2017-08-04,2479.0,2369.0,2529.0,2574.0,2177.899902,2184.600098
2981289,2017-08-05,2093.0,2006.0,2183.0,2351.0,2162.300049,2191.100098
2983071,2017-08-06,968.0,2859.0,1079.0,1181.0,2209.399902,2207.699951
2984853,2017-08-07,2086.0,2258.0,2381.0,2338.0,2220.100098,2192.899902
2986635,2017-08-08,2418.0,1024.0,2589.0,2372.0,2212.300049,2191.199951
2988417,2017-08-09,2311.0,2158.0,2369.0,2460.0,2180.399902,2188.199951


In [56]:
# Create a clean display copy
rolling_example_display = rolling_example.head(10).copy()

# Select only numeric columns
numeric_columns = rolling_example_display.select_dtypes(
    include="number"
).columns

# Round only numeric values
rolling_example_display[numeric_columns] = (
    rolling_example_display[numeric_columns].round(1)
)

rolling_example_display

,date,sales,sales_lag_16,sales_lag_21,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_28
2972379,2017-07-31,2414.0,2183.0,2338.0,2526.0,2168.300049,2207.500000
2974161,2017-08-01,2627.0,1079.0,2372.0,2284.0,2153.699951,2210.100098
2975943,2017-08-02,2645.0,2381.0,2460.0,2691.0,2159.899902,2206.100098
2977725,2017-08-03,2037.0,2589.0,2115.0,2284.0,2190.899902,2192.199951
2979507,2017-08-04,2479.0,2369.0,2529.0,2574.0,2177.899902,2184.600098
2981289,2017-08-05,2093.0,2006.0,2183.0,2351.0,2162.300049,2191.100098
2983071,2017-08-06,968.0,2859.0,1079.0,1181.0,2209.399902,2207.699951
2984853,2017-08-07,2086.0,2258.0,2381.0,2338.0,2220.100098,2192.899902
2986635,2017-08-08,2418.0,1024.0,2589.0,2372.0,2212.300049,2191.199951
2988417,2017-08-09,2311.0,2158.0,2369.0,2460.0,2180.399902,2188.199951


In [57]:
# Select validation targets and historical features
validation_results = model_data.loc[
    validation_mask,
    [
        "sales",
        "sales_lag_21",
        "sales_lag_28",
        "sales_rolling_mean_7",
        "sales_rolling_mean_28"
    ]
].copy()

# Use historical values as simple forecasts
validation_results["baseline_lag_21"] = (
    validation_results["sales_lag_21"]
)

validation_results["baseline_lag_28"] = (
    validation_results["sales_lag_28"]
)

# Average two same-weekday historical values
validation_results["baseline_weekly_average"] = (
    validation_results[
        ["sales_lag_21", "sales_lag_28"]
    ].mean(axis=1)
)

validation_results.head()

,sales,sales_lag_21,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_28,baseline_lag_21,baseline_lag_28,baseline_weekly_average
2972376,8.0,3.0,0.0,5.428571,4.535714,3.0,0.0,1.5
2972377,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0
2972378,3.0,3.0,5.0,4.142857,4.071429,3.0,5.0,4.0
2972379,2414.0,2338.0,2526.0,2168.285645,2207.464355,2338.0,2526.0,2432.0
2972380,1.0,0.0,1.0,0.428571,0.214286,0.0,1.0,0.5


In [58]:
# Calculate forecasting metrics
def calculate_forecast_metrics(actual, predicted):
    actual = actual.astype("float64")
    predicted = predicted.astype("float64").clip(lower=0)

    absolute_error = np.abs(actual - predicted)

    mae = absolute_error.mean()

    wape_pct = (
        absolute_error.sum()
        / actual.sum()
        * 100
    )

    rmsle = np.sqrt(
        np.mean(
            (
                np.log1p(actual)
                - np.log1p(predicted)
            ) ** 2
        )
    )

    return {
        "mae": mae,
        "wape_pct": wape_pct,
        "rmsle": rmsle
    }


# List the baseline columns to evaluate
baseline_columns = [
    "baseline_lag_21",
    "baseline_lag_28",
    "baseline_weekly_average",
    "sales_rolling_mean_7",
    "sales_rolling_mean_28"
]

baseline_metrics_list = []

# Evaluate every baseline separately
for baseline_name in baseline_columns:
    metrics = calculate_forecast_metrics(
        actual=validation_results["sales"],
        predicted=validation_results[baseline_name]
    )

    metrics["baseline"] = baseline_name
    baseline_metrics_list.append(metrics)

# Convert the results into a table
baseline_metrics = pd.DataFrame(
    baseline_metrics_list
)[
    ["baseline", "mae", "wape_pct", "rmsle"]
]

baseline_metrics.sort_values("rmsle").round(3)

,baseline,mae,wape_pct,rmsle
4,sales_rolling_mean_28,99.041,21.201,0.550
3,sales_rolling_mean_7,98.624,21.112,0.558
2,baseline_weekly_average,81.398,17.425,0.575
1,baseline_lag_28,82.871,17.740,0.627
0,baseline_lag_21,92.716,19.848,0.633


In [59]:
%pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [60]:
import sklearn

from sklearn.ensemble import HistGradientBoostingRegressor

print(f"Scikit-learn version: {sklearn.__version__}")
print("HistGradientBoostingRegressor imported successfully")

Scikit-learn version: 1.9.0
HistGradientBoostingRegressor imported successfully


In [61]:
# Define categorical features
categorical_features = [
    "store_nbr",
    "family",
    "city",
    "state",
    "type",
    "cluster",
    "month",
    "day_of_week"
]

# Define numerical features
numerical_features = [
    "year",
    "day_of_month",
    "week_of_year",
    "is_weekend",
    "onpromotion",
    "oil_price",
    "is_national_holiday",
    "is_regional_holiday",
    "is_local_holiday",
    "is_holiday",
    "is_event",
    "is_special_work_day",
    "sales_lag_16",
    "sales_lag_21",
    "sales_lag_28",
    "sales_rolling_mean_7",
    "sales_rolling_mean_28"
]

feature_columns = (
    categorical_features
    + numerical_features
)

print(f"Categorical features: {len(categorical_features)}")
print(f"Numerical features: {len(numerical_features)}")
print(f"Total features: {len(feature_columns)}")

Categorical features: 8
Numerical features: 17
Total features: 25


In [62]:
# Convert categorical features to the pandas category type
for column in categorical_features:
    model_data[column] = model_data[column].astype("category")

print(model_data[categorical_features].dtypes)

store_nbr      category
family         category
city           category
state          category
type           category
cluster        category
month          category
day_of_week    category
dtype: object


In [63]:
# Use recent history for the first model
model_training_start = pd.Timestamp("2015-01-01")

model_training_mask = (
    training_mask
    & (model_data["date"] >= model_training_start)
)

# Create training features and target
X_train = model_data.loc[
    model_training_mask,
    feature_columns
].copy()

y_train = model_data.loc[
    model_training_mask,
    "sales"
].copy()

# Create validation features and target
X_validation = model_data.loc[
    validation_mask,
    feature_columns
].copy()

y_validation = model_data.loc[
    validation_mask,
    "sales"
].copy()

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

print(f"X_validation shape: {X_validation.shape}")
print(f"y_validation shape: {y_validation.shape}")

print(
    f"Missing training features: "
    f"{X_train.isna().sum().sum():,}"
)

print(
    f"Missing validation features: "
    f"{X_validation.isna().sum().sum():,}"
)

print(
    f"Negative training targets: "
    f"{(y_train < 0).sum():,}"
)

X_train shape: (1675080, 25)
y_train shape: (1675080,)
X_validation shape: (28512, 25)
y_validation shape: (28512,)
Missing training features: 0
Missing validation features: 0
Negative training targets: 0


In [64]:
from time import perf_counter

# Create the first machine learning model
sales_model = HistGradientBoostingRegressor(
    loss="poisson",
    learning_rate=0.08,
    max_iter=100,
    max_leaf_nodes=31,
    min_samples_leaf=100,
    l2_regularization=1.0,
    categorical_features="from_dtype",
    early_stopping=False,
    random_state=42,
    verbose=1
)

# Measure the training time
training_start_time = perf_counter()

sales_model.fit(
    X_train,
    y_train
)

training_seconds = (
    perf_counter() - training_start_time
)

print(
    f"Training time: "
    f"{training_seconds / 60:.1f} minutes"
)

Binning 0.335 GB of training data: 0.173 s
Fitting gradient boosted rounds:
Fit 100 trees in 5.536 s, (3100 total leaves)
Time spent computing histograms: 2.993s
Time spent finding best splits:  0.093s
Time spent applying splits:      0.409s
Time spent predicting:           0.105s
Training time: 0.1 minutes


In [65]:
# Generate validation predictions
model_predictions = sales_model.predict(
    X_validation
)

# Prevent negative forecasts
model_predictions = np.clip(
    model_predictions,
    a_min=0,
    a_max=None
)

print(f"Predictions: {len(model_predictions):,}")
print(f"Minimum prediction: {model_predictions.min():.2f}")
print(f"Maximum prediction: {model_predictions.max():.2f}")
print(f"Average prediction: {model_predictions.mean():.2f}")
print(f"Average actual sales: {y_validation.mean():.2f}")

Predictions: 28,512
Minimum prediction: 0.38
Maximum prediction: 15991.14
Average prediction: 482.54
Average actual sales: 467.14


In [68]:
# Calculate machine learning model metrics
model_metrics = calculate_forecast_metrics(
    actual=y_validation,
    predicted=model_predictions
)

# Convert model metrics into a table
model_metrics_table = pd.DataFrame({
    "model": ["HistGradientBoosting"],
    "mae": [model_metrics["mae"]],
    "wape_pct": [model_metrics["wape_pct"]],
    "rmsle": [model_metrics["rmsle"]]
})

# Prepare baseline metrics for comparison
baseline_comparison = baseline_metrics.rename(
    columns={"baseline": "model"}
)

# Combine baseline and machine learning results
metric_comparison = pd.concat(
    [
        baseline_comparison,
        model_metrics_table
    ],
    ignore_index=True
)

# Display models from best to worst RMSLE
metric_comparison = metric_comparison.sort_values(
    "rmsle"
).reset_index(drop=True)

metric_comparison.round(3)

,model,mae,wape_pct,rmsle
0,HistGradientBoosting,76.160,16.303,0.479
1,sales_rolling_mean_28,99.041,21.201,0.550
2,sales_rolling_mean_7,98.624,21.112,0.558
3,baseline_weekly_average,81.398,17.425,0.575
4,baseline_lag_28,82.871,17.740,0.627
5,baseline_lag_21,92.716,19.848,0.633


In [69]:
# Create a detailed validation results table
validation_evaluation = model_data.loc[
    validation_mask,
    [
        "date",
        "store_nbr",
        "family",
        "sales"
    ]
].copy()

# Add model predictions
validation_evaluation["prediction"] = (
    model_predictions.astype("float32")
)

# Calculate signed forecast error
validation_evaluation["forecast_error"] = (
    validation_evaluation["prediction"]
    - validation_evaluation["sales"]
)

# Calculate absolute forecast error
validation_evaluation["absolute_error"] = (
    validation_evaluation["forecast_error"].abs()
)

validation_evaluation.head()

,date,store_nbr,family,sales,prediction,forecast_error,absolute_error
2972376,2017-07-31,1,AUTOMOTIVE,8.0,5.366275,-2.633725,2.633725
2972377,2017-07-31,1,BABY CARE,0.0,0.380927,0.380927,0.380927
2972378,2017-07-31,1,BEAUTY,3.0,4.885498,1.885498,1.885498
2972379,2017-07-31,1,BEVERAGES,2414.0,2255.722900,-158.277100,158.277100
2972380,2017-07-31,1,BOOKS,1.0,0.380927,-0.619073,0.619073


In [70]:
# Calculate total validation volumes
total_actual_sales = (
    validation_evaluation["sales"].sum()
)

total_predicted_sales = (
    validation_evaluation["prediction"].sum()
)

# Calculate overall forecast bias
forecast_bias_pct = (
    (
        total_predicted_sales
        - total_actual_sales
    )
    / total_actual_sales
    * 100
)

print(f"Actual sales: {total_actual_sales:,.0f}")
print(f"Predicted sales: {total_predicted_sales:,.0f}")
print(f"Forecast bias: {forecast_bias_pct:.2f}%")

Actual sales: 13,319,180
Predicted sales: 13,758,307
Forecast bias: 3.30%


In [71]:
# Summarize validation errors by product family
family_error_summary = (
    validation_evaluation
    .groupby(
        "family",
        observed=True,
        as_index=False
    )
    .agg(
        actual_sales=("sales", "sum"),
        predicted_sales=("prediction", "sum"),
        absolute_error=("absolute_error", "sum"),
        row_count=("sales", "size")
    )
)

# Calculate signed forecast bias in sales units
family_error_summary["forecast_bias_units"] = (
    family_error_summary["predicted_sales"]
    - family_error_summary["actual_sales"]
)

# Avoid division by zero
family_actual_denominator = (
    family_error_summary["actual_sales"]
    .replace(0, np.nan)
)

# Calculate bias as a percentage
family_error_summary["forecast_bias_pct"] = (
    family_error_summary["forecast_bias_units"]
    / family_actual_denominator
    * 100
)

# Calculate WAPE for each family
family_error_summary["wape_pct"] = (
    family_error_summary["absolute_error"]
    / family_actual_denominator
    * 100
)

# Calculate each family's contribution to total absolute error
family_error_summary["error_contribution_pct"] = (
    family_error_summary["absolute_error"]
    / family_error_summary["absolute_error"].sum()
    * 100
)

# Sort by contribution to the total model error
family_error_summary = family_error_summary.sort_values(
    "error_contribution_pct",
    ascending=False
).reset_index(drop=True)

family_error_summary.head(15).round(1)

,family,actual_sales,predicted_sales,absolute_error,row_count,forecast_bias_units,forecast_bias_pct,wape_pct,error_contribution_pct
0,GROCERY I,3.980412e+06,4.009923e+06,552650.375000,864,29510.199219,0.700000,13.900000,25.5
1,BEVERAGES,3.005407e+06,3.118602e+06,544909.500000,864,113194.796875,3.800000,18.100000,25.1
2,PRODUCE,1.977480e+06,2.131834e+06,255297.093750,864,154353.593750,7.800000,12.900000,11.8
3,CLEANING,1.065243e+06,1.096090e+06,231663.796875,864,30847.199219,2.900000,21.700001,10.7
4,DAIRY,7.210340e+05,7.779888e+05,86983.296875,864,56954.800781,7.900000,12.100000,4.0
5,BREAD/BAKERY,4.667955e+05,4.792872e+05,59963.800781,864,12491.700195,2.700000,12.800000,2.8
6,HOME CARE,2.518930e+05,2.833844e+05,54945.601562,864,31491.400391,12.500000,21.799999,2.5
7,MEATS,3.210705e+05,3.267790e+05,47378.601562,864,5708.399902,1.800000,14.800000,2.2
8,PERSONAL CARE,2.776310e+05,2.696816e+05,46288.000000,864,-7949.399902,-2.900000,16.700001,2.1
9,POULTRY,3.235603e+05,3.417190e+05,45213.699219,864,18158.800781,5.600000,14.000000,2.1
